# Task

**Row reduction.** Implement a function that computes the reduced row echelon form of a matrix and its pivot columns.

```python
def rref(A, tol=1e-10):
    """Return (R, pivots): the RREF of A and the list of pivot columns."""
    R = np.array(A, dtype=float)  # work on a copy of A
    ...
```

Go through the columns from left to right, keeping track of the row $r$ in which the next pivot will be placed. For each column $c$, find the entry with the largest absolute value among the rows $r, r+1, \ldots$ of that column. If it is smaller than `tol`, the column has no pivot. Otherwise, swap its row with row $r$, divide row $r$ by the pivot, and subtract multiples of row $r$ from all the other rows so that all other entries of column $c$ become zero. (Choosing the largest available entry as the pivot is called partial pivoting; it reduces rounding errors.)
- Test your function: the matrix $\begin{pmatrix} 1 & 2 & -1 \\ 3 & -1 & 4 \end{pmatrix}$ should give $\begin{pmatrix} 1 & 0 & 1 \\ 0 & 1 & -1 \end{pmatrix}$ with pivots `[0, 1]`, and the matrix $F$ of Example (see `ex:F` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)) should give $\begin{pmatrix} 1 & 0 & -1 \\ 0 & 1 & 2 \\ 0 & 0 & 0 \end{pmatrix}$.
- Using `rref`, write the functions `rank(A)` and `null_space_basis(A)`, which returns one basis vector for each free variable as in the proof of the rank--nullity theorem, and `col_space_basis(A)`, which returns the pivot columns of the *original* matrix (Theorem (see `thm:rowops` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf))).
- For $20$ random $5 \times 7$ matrices $A$, check that `A @ v` is close to zero for every vector `v` in `null_space_basis(A)`, that `rank(A) + len(null_space_basis(A)) == 7` (rank--nullity), and that `rank(A)` agrees with `np.linalg.matrix_rank(A)`.
- Random matrices almost always have the largest possible rank. Construct a $6 \times 8$ matrix of rank $4$ as the product of a random $6 \times 4$ matrix and a random $4 \times 8$ matrix, and check that your functions find rank $4$ and a $4$-dimensional null space.


In [1]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

In [2]:
def rref(A, tol=1e-10):
    """Return the reduced row echelon form and zero-based pivot columns."""
    R = np.array(A, dtype=float, copy=True)
    if R.ndim != 2:
        raise ValueError("A must be a two-dimensional array")
    m, n = R.shape
    pivots = []
    row = 0
    for col in range(n):
        if row == m:
            break
        pivot = row + np.argmax(np.abs(R[row:, col]))
        if abs(R[pivot, col]) < tol:
            continue
        R[[row, pivot]] = R[[pivot, row]]
        R[row] /= R[row, col]
        for i in range(m):
            if i != row:
                R[i] -= R[i, col] * R[row]
        pivots.append(col)
        row += 1
    R[np.abs(R) < tol] = 0.0
    return R, pivots

In [3]:
# Check the examples.
A = np.array([[1, 2, -1], [3, -1, 4]])
F = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

for matrix in (A, F):
    R, pivots = rref(matrix)
    print("RREF:\n", R)
    print("Pivot columns:", pivots, "\n")


RREF:
 [[ 1.  0.  1.]
 [ 0.  1. -1.]]
Pivot columns: [0, 1] 

RREF:
 [[ 1.  0. -1.]
 [ 0.  1.  2.]
 [ 0.  0.  0.]]
Pivot columns: [0, 1] 



In [4]:
def rank(A, tol=1e-10):
    return len(rref(A, tol)[1])


def null_space_basis(A, tol=1e-10):
    R, pivots = rref(A, tol)
    free = [j for j in range(R.shape[1]) if j not in pivots]
    basis = []
    for j in free:
        v = np.zeros(R.shape[1])
        # Set this free variable to 1 and the others to 0.
        v[j] = 1
        for i, pivot in enumerate(pivots):
            v[pivot] = -R[i, j]
        basis.append(v)
    return basis


def col_space_basis(A, tol=1e-10):
    A = np.asarray(A)
    pivots = rref(A, tol)[1]
    return A[:, pivots]

In [5]:
# Check rank and null space on random matrices.
for i in range(20):
    A = rng.normal(size=(5, 7))
    basis = null_space_basis(A)
    residuals = [np.linalg.norm(A @ v) for v in basis]
    print(f"Matrix {i + 1}: rank = {rank(A)}, "
          f"NumPy rank = {np.linalg.matrix_rank(A)}, "
          f"rank + nullity = {rank(A) + len(basis)}, "
          f"largest null-space residual = {max(residuals, default=0):.2e}")


Matrix 1: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 7.45e-16
Matrix 2: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 3.55e-16
Matrix 3: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 2.19e-15
Matrix 4: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 5.55e-16
Matrix 5: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 2.36e-15
Matrix 6: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 2.02e-15
Matrix 7: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 5.09e-16
Matrix 8: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 8.02e-13
Matrix 9: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 1.84e-15
Matrix 10: rank = 5, NumPy rank = 5, rank + nullity = 7, largest null-space residual = 4.86e-15
Matrix 11: rank = 5, NumPy rank = 5, rank + nulli

In [6]:
# Check a matrix of rank 4.
A = rng.normal(size=(6, 4)) @ rng.normal(size=(4, 8))
basis = null_space_basis(A)
print("Rank:", rank(A))
print("Null-space dimension:", len(basis))
print("Null-space residuals:", [np.linalg.norm(A @ v) for v in basis])
print("Column-space basis (columns):\n", col_space_basis(A))


Rank: 4
Null-space dimension: 4
Null-space residuals: [np.float64(2.039657145912241e-15), np.float64(3.763312163380282e-15), np.float64(4.042630793071112e-15), np.float64(2.434909185329438e-15)]
Column-space basis (columns):
 [[-2.17306  0.49878  1.29777  2.74944]
 [-2.6236   0.13582  0.96078  2.1771 ]
 [ 0.28526 -1.0881   0.13945 -0.25221]
 [ 1.18954 -0.67446 -0.09475 -0.04042]
 [ 1.84764 -1.91702  0.0954  -0.13261]
 [-1.02064 -1.72443  2.59721  4.79425]]
